# 05 · Differential privacy

Federated learning keeps ECGs inside each hospital, but the model updates that
leave it can still leak information about individual patients. Differential
privacy bounds that leak. This notebook measures what the bound costs.

**The guarantee.** A training algorithm is (ε, δ)-differentially private if
adding or removing any single ECG changes the probability of any outcome by at
most a factor e^ε, except with probability δ. Smaller ε is more private; ε = 1
is strong, ε = 8 is common in practice. δ is fixed at 10⁻⁵, below one over
the number of records every data owner holds.

**DP-SGD** (Abadi et al., 2016) makes training private by changing each step:

1. Batches are drawn by Poisson sampling: each record joins a batch
   independently, with probability batch size / records.
2. Every record's gradient is clipped to L2 norm 1, so no single ECG can move
   the model far.
3. Gaussian noise is added to the summed gradient, scaled to that bound.

The noise level is set before training so that the whole planned run spends
exactly the target ε (Opacus's PRV accountant). Picking the best epoch on the
validation fold is post-processing, which costs no privacy, because
validation records are not training data.

Prerequisite (about 50 minutes on a laptop GPU):

```bash
for c in dp_none dp_eps8 dp_eps3 dp_eps1; do
  uv run python scripts/train_centralized.py --config $c.yaml
done
for c in fed_dp_none fed_dp_eps8 fed_dp_eps3 fed_dp_eps1; do
  uv run python scripts/train_federated.py --config $c.yaml
done
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from fedecg.paths import TABLES_DIR

experiments = pd.read_csv(TABLES_DIR / "experiments.csv")
baseline = experiments.set_index("run").loc["centralized", "macro_auroc"]
dp = experiments[experiments["phase"] == 6].copy()
dp["setting_group"] = dp["algorithm"].map(
    lambda a: "centralized" if a == "centralized" else "federated"
)

## The recipe, and why it has its own control

DP-SGD wants fewer, larger steps: noise is added once per step, so a bigger
batch shares it among more records. The recipe was chosen at ε = 8 on the
validation fold only (the hyperparameter search itself is not privacy
accounted, as is usual in DP papers):

| Batch | Learning rate | Val macro AUROC at ε = 8 |
|---|---|---|
| 256 | 1e-3 | 0.8629 |
| 256 | 3e-3 | 0.8687 |
| 256 | 1e-2 | 0.8731 |
| 256 | 3e-2 | 0.8709 |
| 1024 | 1e-2 | 0.8757 |
| **1024** | **3e-2** | **0.8770** |

Batch 1,024 at a learning rate of 0.03 is far from the non-private recipe
(batch 64, 0.001), and it is worse without privacy. So every DP run has a
**control**: the same recipe with clipping and noise switched off. The gap to
the control is the price of privacy; the gap from the control to the phase 3
baseline is the price of the recipe. The federated runs keep batches of 256
per hospital, since a hospital with 3,400 records would otherwise take only
three steps per round.

In [ ]:
columns = ["setting", "epsilon", "macro_auroc", "macro_f1", "best_step"]
control = dp[dp["epsilon"].isna()].set_index("setting_group")["macro_auroc"]
dp["vs_control"] = dp["macro_auroc"] - dp["setting_group"].map(control)
dp["vs_baseline"] = dp["macro_auroc"] - baseline
dp.sort_values(["setting_group", "epsilon"], na_position="last")[
    [*columns, "vs_control", "vs_baseline"]
].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = [1, 3, 8, None]
labels = ["1", "3", "8", "no privacy"]
for group, name in [("centralized", "centralized"), ("federated", "FedAvg, 5 hospitals")]:
    rows = dp[dp["setting_group"] == group]
    values = [
        rows.loc[
            rows["epsilon"].isna() if eps is None else rows["epsilon"].round() == eps, "macro_auroc"
        ].mean()
        for eps in order
    ]
    ax.plot(range(len(order)), values, marker="o", label=name)
ax.axhline(baseline, color="0.4", linewidth=1, label="phase 3 baseline")
ax.set_xticks(range(len(order)), labels)
ax.set_xlabel("privacy budget epsilon (delta = 1e-5)")
ax.set_ylabel("test macro AUROC")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()

## Which classes pay for privacy?

In [ ]:
per_class = dp.set_index("setting")[[f"auroc_{c}" for c in ["NORM", "MI", "STTC", "CD", "HYP"]]]
per_class.round(3)

## What this means

- **Privacy has a clear, steep price.** Centrally, against the same recipe
  without privacy (0.898): ε = 8 costs 0.024 AUROC, ε = 3 costs 0.035, and
  ε = 1 costs 0.066. All far above the ~0.002 seed noise measured in
  notebook 02.
- **Against the phase 3 baseline the gap is larger**, 0.043 to 0.085, because
  the DP recipe itself gives up 0.019 even without noise: batches of 1,024,
  30 epochs, and the 32-channel network. DP keeps the smaller network because
  per-record gradients of the tuned 64-channel one at batch 1,024 do not fit a
  laptop's memory, and noise added in every parameter's direction tends to
  hurt wider networks more.
- **Federated DP pays more, increasingly so at small ε.** Five hospitals, each
  with DP-SGD at the same ε, lose 0.027 (ε = 8), 0.046 (ε = 3) and 0.112
  (ε = 1) against their own no-privacy control (0.867): 1.1, 1.3 and 1.7
  times the centralized cost. Each hospital calibrates its noise to its own
  ~3,400 records, a fifth of the centralized training set, and DP noise weighs
  more the fewer records share it. At ε = 1 the model was still improving at
  round 30.
- **The rare, subtle classes pay most.** At ε = 1 centrally, MI loses 0.107 and
  CD 0.097 against NORM's 0.029. Clipping caps how much any single record can
  teach the model, which hurts most where examples are fewer and the signal is
  smaller.
- **What the guarantee covers.** Record-level privacy within each data owner:
  one ECG's influence on the released model is bounded. It does not hide
  whether a hospital took part; that needs noise added at aggregation instead.